In [2]:
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = (SparkSession
       .builder
       .appName("DF SF Fire Example")
       .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 13:50:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Reading structured data using a schema 

Spark can infer the schema but that can be costly. Spark can also infer schema from a sample at a lesser cost. Here we will define the schema explicitly. Note that we changed column names to not have space in them as that can be problematic when saving data.

In [4]:
fire_schema =  StructType([StructField('CallNumber', IntegerType(), True),
                           StructField('UnitID', StringType(), True),
                           StructField('IncidentNumber', IntegerType(), True),
                           StructField('CallType', StringType(), True),
                           StructField('CallDate', StringType(), True),
                           StructField('WatchDate', StringType(), True),
                           StructField('CallFinalDisposition', StringType(), True),
                           StructField('AvailableDtTm', StringType(), True),
                           StructField('Address', StringType(), True),
                           StructField('City', StringType(), True),
                           StructField('Zipcode', IntegerType(), True),
                           StructField('Battalion', StringType(), True),
                           StructField('StationArea', StringType(), True),
                           StructField('Box', StringType(), True),
                           StructField('OriginalPriority', StringType(), True),
                           StructField('Priority', StringType(), True),
                           StructField('FinalPriority', IntegerType(), True),
                           StructField('ALSUnit', BooleanType(), True),
                           StructField('CallTypeGroup', StringType(), True),
                           StructField('NumAlarms', IntegerType(), True),
                           StructField('UnitType', StringType(), True),
                           StructField('UnitSequenceInCallDispatch', IntegerType(), True),
                           StructField('FirePreventionDistrict', StringType(), True),
                           StructField('SupervisorDistrict', StringType(), True),
                           StructField('Neighborhood', StringType(), True),
                           StructField('Location', StringType(), True),
                           StructField('RowID', StringType(), True),
                           StructField('Delay', FloatType(), True)])

In [5]:
sf_fire_file = "sf-fire/sf-fire-calls.csv"
fire_df = spark.read.csv(sf_fire_file, header=True, schema=fire_schema)

# Projections and Filters

A *projection* in relation parlance is a way to return only the rows matching a certain realtional condition by using filters. In Spark, projections are done with the `select()` method, while filters can be expressed using the `filter()` or `where()` methods.

In [6]:
few_fire_df = (fire_df.select("IncidentNumber", "AvailableDtTm", "CallType")
                .where(col("CallType") != "Medical Incident"))

In [7]:
few_fire_df.show(5,False)

+--------------+----------------------+--------------+
|IncidentNumber|AvailableDtTm         |CallType      |
+--------------+----------------------+--------------+
|2003235       |01/11/2002 01:51:44 AM|Structure Fire|
|2003250       |01/11/2002 04:16:46 AM|Vehicle Fire  |
|2003259       |01/11/2002 06:01:58 AM|Alarms        |
|2003279       |01/11/2002 08:03:26 AM|Structure Fire|
|2003301       |01/11/2002 09:46:44 AM|Alarms        |
+--------------+----------------------+--------------+
only showing top 5 rows


What if we want to know how many distinct `CallTypes` were recorded as the causes of fire calls?

In [8]:
(fire_df
 .select("CallType")
 .where(col("CallType").isNotNull())
 .agg(count_distinct("CallType").alias("DistinctCallTypes"))
 .show(truncate=False))
      

[Stage 1:=============================>                             (2 + 2) / 4]

+-----------------+
|DistinctCallTypes|
+-----------------+
|30               |
+-----------------+



We can list the distinct call types in the daya set using the following query.

In [9]:
(fire_df
 .select("CallType")
 .where(col("CallType").isNotNull())
 .distinct()
 .show(truncate=False))

+--------------------------------------------+
|CallType                                    |
+--------------------------------------------+
|Elevator / Escalator Rescue                 |
|Marine Fire                                 |
|Aircraft Emergency                          |
|Confined Space / Structure Collapse         |
|Administrative                              |
|Alarms                                      |
|Odor (Strange / Unknown)                    |
|Citizen Assist / Service Call               |
|HazMat                                      |
|Watercraft in Distress                      |
|Explosion                                   |
|Oil Spill                                   |
|Vehicle Fire                                |
|Suspicious Package                          |
|Extrication / Entrapped (Machinery, Vehicle)|
|Other                                       |
|Outside Fire                                |
|Traffic Collision                           |
|Assist Polic

# Renaming, Adding, and Dropping Columns

We can rename columns for reasons of convention, or sometimes for readability or brevity. Let's change the name of our `Delayed` column to `ResponseDelayedinMins` and take a look at response times that were longer than five minutes.

In [10]:
fire_df.cache()

26/10/08 13:50:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[CallNumber: int, UnitID: string, IncidentNumber: int, CallType: string, CallDate: string, WatchDate: string, CallFinalDisposition: string, AvailableDtTm: string, Address: string, City: string, Zipcode: int, Battalion: string, StationArea: string, Box: string, OriginalPriority: string, Priority: string, FinalPriority: int, ALSUnit: boolean, CallTypeGroup: string, NumAlarms: int, UnitType: string, UnitSequenceInCallDispatch: int, FirePreventionDistrict: string, SupervisorDistrict: string, Neighborhood: string, Location: string, RowID: string, Delay: float]

In [11]:
new_fire_df = fire_df.withColumnRenamed("Delay", "ResponseDelayedinMins")

In [12]:
(new_fire_df
 .select("ResponseDelayedinMins")
 .where(col("ResponseDelayedinMins") > 5)
 .show(5, False))

[Stage 10:>                                                         (0 + 1) / 1]

+---------------------+
|ResponseDelayedinMins|
+---------------------+
|5.35                 |
|6.25                 |
|5.2                  |
|5.6                  |
|7.25                 |
+---------------------+
only showing top 5 rows


In [13]:
fire_ts_df = (new_fire_df
              .withColumn("IncidentDate", to_timestamp(col("CallDate"), "MM/dd/yyyy"))
              .drop("CallDate")
              .withColumn("OnWatchDate", to_timestamp(col("WatchDate"), "MM/dd/yyyy"))
              .drop("WatchDate")
              .withColumn("AvailableDtTS", to_timestamp(col("AvailableDtTm"), "MM/dd/yyyy hh:mm:ss a"))
              .drop("AvailableDtTm"))
              

In [14]:
(fire_ts_df
 .select("IncidentDate", "OnWatchDate", "AvailableDtTS")
         .show(5, False))

+-------------------+-------------------+-------------------+
|IncidentDate       |OnWatchDate        |AvailableDtTS      |
+-------------------+-------------------+-------------------+
|2002-01-11 00:00:00|2002-01-10 00:00:00|2002-01-11 01:51:44|
|2002-01-11 00:00:00|2002-01-10 00:00:00|2002-01-11 03:01:18|
|2002-01-11 00:00:00|2002-01-10 00:00:00|2002-01-11 02:39:50|
|2002-01-11 00:00:00|2002-01-10 00:00:00|2002-01-11 04:16:46|
|2002-01-11 00:00:00|2002-01-10 00:00:00|2002-01-11 06:01:58|
+-------------------+-------------------+-------------------+
only showing top 5 rows


Now we can query using functions from `spark.sql.functions` like `dayofmonth(), dayofyear, dayofweek()` to explore. For example, we could find out how many calls were logged in the last seven days, or we could see how many years' worth of calls are included in the data set.

In [15]:
(fire_ts_df
  .select(year('IncidentDate'))
 .distinct()
 .orderBy(year('IncidentDate'))
 .show())

[Stage 12:===========================================>              (3 + 1) / 4]

+------------------+
|year(IncidentDate)|
+------------------+
|              2000|
|              2001|
|              2002|
|              2003|
|              2004|
|              2005|
|              2006|
|              2007|
|              2008|
|              2009|
|              2010|
|              2011|
|              2012|
|              2013|
|              2014|
|              2015|
|              2016|
|              2017|
|              2018|
+------------------+



# Aggregations

What if we want to know what the most common types of fire calls were, or what zip codes accounted for the most calls?

A handful of transformations and actions on DataFrames such as `groupBy()`, `orderBy()`, and `count()`, offer the ability to aggregate by column names and then aggregate counts across them.

In [16]:
(fire_ts_df
 .select("callType")
 .where(col("CallType").isNotNull())
 .groupBy("callType")
 .count()
 .orderBy("count", ascending=False)
 .show(truncate=False))

+-------------------------------+------+
|callType                       |count |
+-------------------------------+------+
|Medical Incident               |113794|
|Structure Fire                 |23319 |
|Alarms                         |19406 |
|Traffic Collision              |7013  |
|Citizen Assist / Service Call  |2524  |
|Other                          |2166  |
|Outside Fire                   |2094  |
|Vehicle Fire                   |854   |
|Gas Leak (Natural and LP Gases)|764   |
|Water Rescue                   |755   |
|Odor (Strange / Unknown)       |490   |
|Electrical Hazard              |482   |
|Elevator / Escalator Rescue    |453   |
|Smoke Investigation (Outside)  |391   |
|Fuel Spill                     |193   |
|HazMat                         |124   |
|Industrial Accidents           |94    |
|Explosion                      |89    |
|Train / Rail Incident          |57    |
|Aircraft Emergency             |36    |
+-------------------------------+------+
only showing top

DataFrames also provide common statistical functions such as `min(), max(), sum()`, and `avg()`. Here is an example. 

In [17]:
import pyspark.sql.functions as F

In [18]:
(fire_ts_df
 .select(F.sum("NumAlarms"), F.avg("ResponseDelayedinMins"),
         F.min("ResponseDelayedinMins"), F.max("ResponseDelayedinMins"))
 .show())

+--------------+--------------------------+--------------------------+--------------------------+
|sum(NumAlarms)|avg(ResponseDelayedinMins)|min(ResponseDelayedinMins)|max(ResponseDelayedinMins)|
+--------------+--------------------------+--------------------------+--------------------------+
|        176170|         3.892364154521585|               0.016666668|                   1844.55|
+--------------+--------------------------+--------------------------+--------------------------+



Other common statistical methods include `stat()`, `describe()`, `correlation()`, 
`covariance()`, `sampleBy()`, `approxQuantile()`, `frequentItems()`, etc.

# Further Data Explorations

 1. What were all the different types of fire calls in 2018?
 2. What months within the year 2018 saw the highest number of fire calls?
 3. Which neighborhood in San Francisco generated the most fire calls in 2018?
 4. Which neighborhoods had the worst response times to fire calls in 2018?
 5. Which week in 2018 had the most fire calls?
 6. Is there a correlation between neighborhood, zip code, and number of fire calls?

Finally, we show how to write the DataFrame out. The parquet format is a common format that is efficient. It stores the schema as metadata, so we don't need to define it when we read the DataFrame back.

In [19]:
(fire_df.write.format("parquet")
 .mode("overwrite")
 .option("compression", "snappy")
 .save("./parquet"))

In [20]:
saved_fire_df = spark.read.format("parquet").load("./parquet")

In [21]:
saved_fire_df.show(2)

+----------+------+--------------+----------------+----------+----------+--------------------+--------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------+--------------------------+----------------------+------------------+--------------------+--------------------+-------------+-----+
|CallNumber|UnitID|IncidentNumber|        CallType|  CallDate| WatchDate|CallFinalDisposition|       AvailableDtTm|             Address|City|Zipcode|Battalion|StationArea| Box|OriginalPriority|Priority|FinalPriority|ALSUnit|CallTypeGroup|NumAlarms|UnitType|UnitSequenceInCallDispatch|FirePreventionDistrict|SupervisorDistrict|        Neighborhood|            Location|        RowID|Delay|
+----------+------+--------------+----------------+----------+----------+--------------------+--------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+------

In [22]:
spark.stop()